In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from datasets import load_dataset

# 1. Tải bộ dữ liệu MACCROBAT từ Hugging Face Hub
maccrobat = load_dataset("ktgiahieu/maccrobat2018_2020")

# 2. Xem tổng quan cấu trúc bộ dữ liệu
print(maccrobat)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/400 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 400
    })
})


In [ ]:
import torch
import torch.nn as nn

train_sentences = maccrobat["train"]["tokens"]
train_labels = maccrobat["train"]["tags"]

len(train_sentences),

# for x, y in zip(train_sentences[0], train_labels[0]):
#     print(x, '->', y)

(400,)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("d4data/biomedical-ner-all")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
!pip install -qq evaluate
!pip install -qq seqeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import evaluate
import numpy as np

# unique_tags = set(label for labels in train_labels for label in labels)

# unique_tags = sorted(list(unique_tags))

# label2id = {unique_tag: i for i, unique_tag in enumerate(unique_tags)}
# id2label = {i: unique_tag for unique_tag, i in label2id.items()}
model = AutoModelForTokenClassification.from_pretrained(
    "d4data/biomedical-ner-all",
)

label2id = model.config.label2id
id2label = model.config.id2label
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    # Lấy ra mảng dự đoán (logits) và mảng đáp án gốc (labels)
    logits, labels = eval_pred

    # Biến logits thành các ID dự đoán (chọn số có xác suất cao nhất)
    predictions = np.argmax(logits, axis=-1)

    # Chuyển đổi từ ID số sang nhãn chữ (ví dụ: 1 -> 'B-Disease'),
    # VÀ CHỈ LẤY những vị trí mà nhãn gốc KHÁC -100

    true_predictions = [
        [p for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [l for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Tính toán các chỉ số Precision, Recall, F1 với seqeval
    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    # Trả về kết quả cho Trainer in ra màn hình
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }



model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [ ]:
class MaccrobatDataset(torch.utils.data.Dataset):
    def __init__(self, tokens, tags, tokenizer, label2id, max_len=512):
        self.tokens = tokens
        self.tags = tags
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        words = self.tokens[idx]
        word_labels = self.tags[idx]

        # Tokenize và align nhãn
        encoding = self.tokenizer(words, is_split_into_words=True, truncation=True, max_length=self.max_len)

        labels = []
        word_ids = encoding.word_ids()
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None or word_idx == previous_word_idx:
                labels.append(-100)
            else:
                # LƯU Ý: Nếu nhãn trong maccrobat không có trong label2id của model, ta để -100
                label_str = word_labels[word_idx]
                labels.append(self.label2id.get(label_str, -100))
            previous_word_idx = word_idx

        return {
            "input_ids": torch.as_tensor(encoding["input_ids"]),
            "attention_mask": torch.as_tensor(encoding["attention_mask"]),
            "labels": torch.as_tensor(labels)
        }

In [ ]:
maccrobat_split = maccrobat['train'].train_test_split(test_size=0.2, seed=42)

train_data = maccrobat_split['train']
test_data = maccrobat_split['test']

train_dataset = MaccrobatDataset(train_data['tokens'], train_data['tags'], tokenizer, label2id)
eval_dataset = MaccrobatDataset(test_data['tokens'], test_data['tags'], tokenizer, label2id)
len(train_dataset), len(eval_dataset)

(320, 80)

In [ ]:
eval_dataset[0]
input_ids = eval_dataset[0]["input_ids"]
attention_mask = eval_dataset[0]["attention_mask"]
labels = eval_dataset[0]["labels"]

assert len(input_ids) == len(attention_mask) == len(labels)
labels

tensor([-100,    0,    3,   46,   46,   46,   46,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0, -100,    0,    0,
           0,    0,    0,    0, -100, -100, -100,    0,    0,    0,    0, -100,
        -100, -100,    0,    0,    0,    0, -100, -100, -100,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0, -100, -100,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0, -100,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
        -100, -100,    0,    0,    0,    0, -100,    0,    0,    0,    0,    0,
           0,    0, -100, -100, -100,    0,    0,    0,    0,    0,    0,    0,
        -100, -100, -100,    0,    0,    0,    0,    0,    0,    0,    0,   20,
          63,   63,   63,   63,   63,   63,   63, -100])

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForTokenClassification

# Sử dụng id2label của chính mô hình
# Để đảm bảo dự đoán của nó khớp với tên nhãn y khoa
id2label = model.config.id2label
label2id = model.config.label2id

# 2. Chuẩn bị Data Collator (Hàm này giúp gom nhóm và padding tự động)
data_collator = DataCollatorForTokenClassification(tokenizer)

# 3. Thiết lập thông số đánh giá (Evaluation Arguments)
eval_args = TrainingArguments(
    output_dir="./eval_results",
    per_device_eval_batch_size=16,
    report_to="none" # Không gửi log đi đâu cả
)

training_args = TrainingArguments(
    output_dir="./maccrobat_model",
    eval_strategy="epoch",    
    learning_rate=2e-5,        
    per_device_train_batch_size=8,   
    per_device_eval_batch_size=8,
    num_train_epochs=10,          
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,      
    report_to="none"
)

# 4. Khởi tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset, 
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.270725,0.911658,0.927099,0.919314,0.960476
2,No log,0.272653,0.900844,0.943299,0.921583,0.961616
3,No log,0.274693,0.910256,0.941090,0.925416,0.965163
4,No log,0.274531,0.909993,0.945508,0.927411,0.965290
5,No log,0.273414,0.921316,0.948454,0.934688,0.967317
6,No log,0.280762,0.918746,0.949190,0.933720,0.966810
7,No log,0.287607,0.917080,0.952872,0.934633,0.966937
8,No log,0.283100,0.917496,0.949926,0.933430,0.966557
9,No log,0.288345,0.918382,0.952872,0.935309,0.966684
10,No log,0.287793,0.919687,0.952872,0.935986,0.966937


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=400, training_loss=0.007749401926994324, metrics={'train_runtime': 316.3014, 'train_samples_per_second': 10.117, 'train_steps_per_second': 1.265, 'total_flos': 104677451366400.0, 'train_loss': 0.007749401926994324, 'epoch': 10.0})

In [ ]:
print(f"Checkpoint tốt nhất đã nạp: {trainer.state.best_model_checkpoint}")

Checkpoint tốt nhất đã nạp: ./maccrobat_model/checkpoint-40


In [ ]:
import torch

tokens = maccrobat['train'][0]['tokens']
print(len(tokens))
tags = maccrobat['train'][0]['tags']
inputs = tokenizer(tokens, truncation=True, is_split_into_words=True, return_tensors="pt").to(model.device)
with torch.no_grad():
  outputs = model(**inputs)

predictions = outputs.logits.argmax(dim=-1)[0].cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(inputs.input_ids[0])

merged_tokens = []
merged_labels = []

current_word = ""
current_label = None
for token, prediction in zip(tokens, predictions):
    # 1. Bỏ qua các token đặc biệt ([CLS], [SEP], [PAD])
    if token in tokenizer.all_special_tokens:
        continue

    # 2. Kiểm tra nếu là sub-word (bắt đầu bằng ##)
    if token.startswith("##"):
        # Ghép phần đuôi vào từ hiện tại (bỏ dấu ##)
        current_word += token[2:]
        # KHÔNG cập nhật current_label (giữ nguyên nhãn của từ gốc)
    else:
        # Nếu đây là một từ mới, hãy lưu từ cũ lại trước đã
        if current_word:
            merged_tokens.append(current_word)
            merged_labels.append(current_label)

        # Bắt đầu khởi tạo từ mới và nhãn gốc của nó
        current_word = token
        current_label = model.config.id2label[prediction]



# Lưu lại từ cuối cùng sau khi kết thúc vòng lặp

if current_word:
    merged_tokens.append(current_word)
    merged_labels.append(current_label)
print(len(merged_tokens))

# 3. In kết quả đã được "hàn gắn"
print(f"{'Từ hoàn chỉnh':<20} | {'Nhãn dự đoán':<25} | Nhãn thật")
print("-" * 70)
for word, label, tag in zip(merged_tokens, merged_labels, tags):
    print(f"{word:<20} | {label:<25} | {tag}")

a -> O
63 -> B-Age
- -> I-Age
year -> I-Age
- -> I-Age
old -> I-Age
female -> B-Sex
with -> O
a -> O
history -> O
of -> O
type -> B-History
2 -> I-History
diabetes -> I-History
presented -> B-Clinical_event
with -> O
a -> O
2 -> B-Duration
- -> I-Duration
day -> I-Duration
history -> O
of -> O
severe -> O
chest -> O
pain -> B-Sign_symptom
and -> O
was -> O
prescribed -> O
as -> B-Medication
##pi -> B-Medication
##rin -> I-Medication
81 -> B-Dosage
##mg -> I-Dosage
once -> I-Dosage
daily -> I-Dosage
. -> O


In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

# 4. Lấy ID nhãn có xác suất cao nhất
predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# 5. Giải mã và in kết quả
print(f"{'Token':<20} | {'Nhãn dự đoán'}")
print("-" * 35)
for token, pred_id in zip(tokens, predictions):
    # Loại bỏ các token đặc biệt [CLS], [SEP], [PAD]
    if token not in tokenizer.all_special_tokens:
        label = model.config.id2label[pred_id]
        print(f"{token:<20} | {label}")